# Ablation campaign — Phase 2A + 2B

Runs every distillation configuration reported in `README.md`, against the
teacher produced by `teacher_training_colab.ipynb` (val2017 mAP 0.142).

| Phase | Runs | Seeds | Purpose |
|---|---|---|---|
| 2A | 0–8 | 42 | The nine-run ablation: baseline, 5 methods, 3 controls |
| 2B λ-swap | 9–10 | 42 | Cross the stage-adaptive λ values to separate schedule direction from weight |
| 2B seeds | 0, 5, 6, 9 | 43, 44 | Repeat the contested comparisons to separate real differences from seed noise |

Each run: R18 student, 30K COCO subset, 36 epochs, 512 px, checkpoint selected
on the 2.5K selection split, val2017 evaluated once at the end. ~3 h per run.

`scripts/run_ablation.sh` pins `--lr-head 1e-4 --lr-backbone 1e-5`. The
`train_kd.py` default of 1e-3 collapses this architecture — see the teacher
notebook.

Runs are resumable: a completed run (checkpoint + eval.log) is skipped, so
re-running a cell after a dropped session continues where it stopped.


## 1 — Runtime

In [ ]:
gpu = !nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
print(gpu[0])
assert 'A100' in gpu[0], f'Expected an A100, got: {gpu[0]}'

## 2 — Drive + teacher checkpoint

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
RUNS    = '/content/drive/MyDrive/rtdetr_runs'
TEACHER = f'{RUNS}/teacher_r50_lr1e4/checkpoint_best.pth'
assert os.path.exists(TEACHER), f'Teacher checkpoint not found: {TEACHER}'
print('teacher:', TEACHER)

## 3 — Repo

In [ ]:
%cd /content
if not os.path.exists('/content/rt-detr-kd'):
    !git clone --recurse-submodules https://github.com/umutonuryasar/rt-detr-kd
%cd /content/rt-detr-kd
!git pull --ff-only
!git log --oneline -3

## 4 — Dependencies

In [ ]:
!pip install -q pycocotools scipy pyyaml tensorboard
import torch; print('torch', torch.__version__, '| cuda', torch.cuda.is_available())
assert torch.cuda.is_available()

## 5 — 30K subset + selection split

The ablation trains on the 30K subset, not full COCO. First run downloads and
samples it (~20 min); afterwards it is cached as a tar on Drive and restored
in ~3 min, so a dropped session does not mean another 18 GB download.

The split is seed-42 stable: the same 27.5K/2.5K partition every time.

In [ ]:
SUB   = '/content/coco_subset'
CACHE = f'{RUNS}/coco_subset.tar'

if os.path.exists(f'{SUB}/train2017_30k'):
    print('subset already on local disk')
elif os.path.exists(CACHE):
    print('restoring from Drive cache...')
    !tar -xf {CACHE} -C /content
else:
    print('no cache — downloading (slow path)')
    !bash scripts/download_coco_subset.sh {SUB}
    !python tools/make_select_split.py \
        --ann {SUB}/annotations/instances_train2017_30k.json \
        --num-select 2500 --seed 42
    !tar -cf {CACHE} -C /content coco_subset
    print('cached to Drive for next time')

n_train = len(os.listdir(f'{SUB}/train2017_30k'))
assert n_train == 30000, f'expected 30000, got {n_train}'
for f in ['instances_train2017_30k_train.json', 'instances_train2017_30k_select.json']:
    assert os.path.exists(f'{SUB}/annotations/{f}'), f'missing {f}'
print('OK — subset + split ready.')

## 6 — λ calibration

Raw KD loss magnitudes differ by orders of magnitude across methods, so a
single `kd_lambda=1.0` would compare methods at wildly different effective KD
strengths. The rule — applied identically to every method — sets
`λ = median(L_det) / median(L_KD)` measured over 20 batches at initialization,
so every KD term *starts* at detection-loss scale.

Re-running this reproduces the values hard-coded in the next cell. It takes
~3 min; skip it if you already trust them.

In [ ]:
!python tools/calibrate_lambda.py \
    --teacher-weights {TEACHER} \
    --coco-train {SUB}/train2017_30k \
    --train-ann {SUB}/annotations/instances_train2017_30k_train.json \
    --batches 20 --batch-size 16 --img-size 512

## 7 — Campaign environment

λ values as measured against this teacher at seed 42. Kept here, visible,
rather than in YAML — a config key silently overriding a CLI flag has bitten
this project before, and `train_kd.py` now refuses to run on that conflict.

In [ ]:
os.environ['TEACHER_WEIGHTS']     = TEACHER
os.environ['BATCH_SIZE']          = '16'
os.environ['IMG_SIZE']            = '512'
os.environ['TEACHER_MIN_MAP']     = '0.10'   # teacher scores 0.142; catches a failed load
os.environ['LAM_LOGIT_BINARY']    = '24.23'
os.environ['LAM_LOGIT_SOFTMAX']   = '5.317'
os.environ['LAM_FEATURE']         = '6.249'
os.environ['LAM_CWD']             = '11.78'
os.environ['LAM_QUERY_HUNGARIAN'] = '3.518'
os.environ['LAM_QUERY_INDEX']     = '3.577'
os.environ['LAM_STAGE_COSINE']    = '6.324'
os.environ['LAM_STAGE_INVCOS']    = '22.51'
print({k: v for k, v in os.environ.items() if k.startswith('LAM_')})

## 8 — Phase 2A: the nine-run ablation (seed 42, ~27 h)

### Check after run 0 (baseline)
- val mAP in a believable band for a 15.9M student on 30K images (this
  campaign: 0.0419).
- Run 1 onward prints the teacher's mAP and clears the 0.10 gate. A run that
  aborts at the gate means the teacher weights did not load — stop and check.
- LR climbs to 1.00e-04 and no scheduler warning appears.

In [ ]:
OUT = f'{RUNS}/ablation'
os.environ['SEED']      = '42'
os.environ['ONLY_RUNS'] = ''          # empty = runs 0-8
!bash scripts/run_ablation.sh {SUB} {OUT} 2>&1 | tee -a {RUNS}/ablation_console.log

## 9 — Phase 2B, λ-swap (runs 9–10, seed 42, ~6 h)

Calibration measures λ at the epoch-1 mixture, which differs between the two
schedule directions: cosine is feature-dominant there (large L_KD → small λ),
inverse-cosine is logit-dominant (small L_KD → large λ). The weights then swap
over training. Crossing the λ values completes a 2×2 that tells schedule
**direction** apart from **weight**:

| | λ=6.324 | λ=22.51 |
|---|---|---|
| cosine | run07 | **run09** |
| inverse-cosine | **run10** | run08 |

In [ ]:
os.environ['SEED']      = '42'
os.environ['ONLY_RUNS'] = '9 10'
!bash scripts/run_ablation.sh {SUB} {OUT} 2>&1 | tee -a {RUNS}/phase2b_lambda.log

## 10 — Phase 2B, seed repeats (seeds 43 + 44, ~24 h)

Only the configurations whose differences carry a claim: baseline (the anchor
and the noisiest), the query matching pair, and the best configuration found.
The stage-adaptive direction pair is not repeated — the λ-swap already showed
the direction effect is ~0.001.

In the original campaign this ran as two passes (`0 5 6`, then `9`); the
combined form below produces the same set.

In [ ]:
import subprocess, time
os.environ['ONLY_RUNS'] = '0 5 6 9'
for seed in ['43', '44']:
    out = f'{RUNS}/ablation_seed{seed}'
    log = f'{RUNS}/phase2b_seed{seed}.log'
    print(f'\n{"="*60}\n SEED {seed} starting {time.strftime("%H:%M:%S")}\n{"="*60}\n', flush=True)
    os.environ['SEED'] = seed
    # subprocess, not `!`: a failure in one seed must not stop the next
    subprocess.run(f'bash scripts/run_ablation.sh {SUB} {out} 2>&1 | tee -a {log}',
                   shell=True, executable='/bin/bash')
    print(f'\n SEED {seed} finished {time.strftime("%H:%M:%S")}', flush=True)

## 11 — Cross-seed results

Reads `results.csv` per seed directory rather than `eval.log`, which rounds
mAP to three decimals — too coarse when the differences under test are ~0.007.

In [ ]:
import subprocess, csv, statistics as st

SEED_DIRS = {42: f'{RUNS}/ablation',
             43: f'{RUNS}/ablation_seed43',
             44: f'{RUNS}/ablation_seed44'}
WANT = ['run00_baseline', 'run05_query_hungarian', 'run06_query_index',
        'run09_stage_cosine_lam22']

for d in SEED_DIRS.values():
    if os.path.isdir(d):
        subprocess.run(f'python tools/aggregate_results.py --runs-dir {d}',
                       shell=True, capture_output=True)

def load(d):
    p = os.path.join(d, 'results.csv')
    if not os.path.exists(p):
        return {}
    out = {}
    with open(p) as f:
        for row in csv.DictReader(f):
            tag = row.get('run') or row.get('tag') or row.get('name')
            for k in ('mAP', 'map', 'mAP@[.5:.95]', 'val_map'):
                if row.get(k):
                    out[tag] = float(row[k]); break
    return out

data = {s: load(d) for s, d in SEED_DIRS.items() if os.path.isdir(d)}

print(f"{'config':<26}" + ''.join(f'seed{s:<7}' for s in data) + '   mean ± std')
print('-' * 76)
means = {}
for tag in WANT:
    vals = [data[s][tag] for s in data if tag in data[s]]
    cells = ''.join(f'{data[s].get(tag, float("nan")):<11.4f}' for s in data)
    if len(vals) >= 2:
        m, sd = st.mean(vals), st.stdev(vals)
        means[tag] = (m, sd)
        print(f'{tag:<26}{cells}   {m:.4f} ± {sd:.4f}  (n={len(vals)})')
    else:
        print(f'{tag:<26}{cells}   (n={len(vals)} — need 2+)')

h, i = means.get('run05_query_hungarian'), means.get('run06_query_index')
if h and i:
    diff   = i[0] - h[0]
    pooled = (h[1]**2 + i[1]**2) ** 0.5
    print(f'\nindex − hungarian = {diff:+.4f}   (pooled sd ≈ {pooled:.4f})')
    print('Verdict:', 'exceeds spread — likely real' if abs(diff) > pooled
          else 'within seed spread — not distinguishable from noise')

## Campaign result

| Configuration | mAP (mean ± std, n=3) |
|---|---|
| Baseline (no KD) | 0.0388 ± 0.0028 |
| Query-KD, Hungarian | 0.0377 ± 0.0007 |
| Query-KD, index | 0.0448 ± 0.0010 |
| Stage-Adaptive, cosine, λ=22.51 | 0.0676 ± 0.0005 |

Findings and limitations are in `README.md`. Note that the best configuration
came from a λ-swap *control* run rather than the calibrated λ, and that no
other method was tried at high λ — so the headline is *the best configuration
found*, not *this is the best method*.